In [1]:
# IN03: API vs MCP, Framework Selection, and Build vs Buy

In [2]:
# Objectives:

# By the end of this notebook you will be able to:
# - Compare REST and MCP integration using live external services
# - Understand why schema standardisation matters at enterprise scale
# - Select between LangChain, LangGraph, and Python-only based on TCO
# - Apply a structured Build vs Buy rubric to any AI tooling decision
# - Produce an ARB-ready framework selection matrix for the Walmart Retail Assistant
# - Live APIs used: OpenAI | OpenWeatherMap | Tavily Search | FastMCP (MCP protocol)

# Deliverable: framework_selection_matrix.txt

In [3]:
# Choosing the right production architecture for a Walmart AI assistant.

# 1. REST API or MCP?
# 2. Which framework should be used?
# 3. Should Walmart build or buy each component?

# Overall business objective: Design a scalable Walmart Retail Assistant that uses live data, has reusable tools, supports complex workflows, controls cost and latency, and is suitable for Architecture Review Board approval.

In [4]:
# Load the libraries and keys needed for the full notebook.
# This cell also defines the store we will use in every example.
import os
import json
import time
import requests
from typing import TypedDict
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)

OPENAI_KEY  = os.getenv('OPENAI_API_KEY')
WEATHER_KEY = os.getenv('OPENWEATHERMAP_API_KEY')
TAVILY_KEY  = os.getenv('TAVILY_API_KEY')

client = OpenAI(api_key=OPENAI_KEY)

# Check whether all required API keys are present before we continue.
missing = [k for k, v in {
    'OPENAI_API_KEY':         OPENAI_KEY,
    'OPENWEATHERMAP_API_KEY': WEATHER_KEY,
    'TAVILY_API_KEY':         TAVILY_KEY,
}.items() if not v]

if missing:
    print(f'WARNING: Missing API keys: {missing}')
    print('Add them to your .env file before running this notebook.')
else:
    print('All required API keys loaded.')

# Fixed store context keeps every comparison in the notebook consistent.
STORE_ID   = 'WMT-2847'
STORE_CITY = 'Bengaluru'
print(f'Store context: {STORE_ID} | {STORE_CITY}, India')

All required API keys loaded.
Store context: WMT-2847 | Bengaluru, India


## The Decision Landscape

Every production AI system at Walmart scale requires three interlocking decisions made before any code is written:

1. **Protocol selection:** How should your AI agent communicate with external systems? REST API or Model Context Protocol (MCP)?
2. **Framework selection:** What orchestration layer do you build on? LangChain, LangGraph, or Python-only?
3. **Build vs Buy:** For each component, is it cheaper to build it or to buy a vendor solution?

Each decision compounds. A wrong protocol choice forces a framework rewrite downstream.

**The running scenario throughout this notebook:**

> *You are the AI engineer for the Walmart India Retail Assistant deployed at 4,700 stores with 50,000+ queries per day. The store manager at WMT-2847 in Bengaluru asks: "Based on today's actual conditions, what should we prioritise stocking today?"*

Every tool call in this notebook returns **live data** from real external APIs. The recommendation changes based on real weather and real market signals retrieved at runtime.

In [5]:
# These helper functions are the live data layer used everywhere below.
# One gets weather data and the other gets demand signals from search.
def fetch_weather(city: str, country_code: str = 'IN') -> dict:
    # Call the live OpenWeatherMap API for current weather.
    url    = 'https://api.openweathermap.org/data/2.5/weather'
    params = {'q': f'{city},{country_code}', 'appid': WEATHER_KEY, 'units': 'metric'}
    resp   = requests.get(url, params=params, timeout=10)
    resp.raise_for_status()
    d = resp.json()
    print("=="*50)
    print(d)
    print("=="*50)
    return {
        'city':           d['name'],
        'country':        d['sys']['country'],
        'temperature_c':  round(d['main']['temp'], 1),
        'feels_like_c':   round(d['main']['feels_like'], 1),
        'humidity_pct':   d['main']['humidity'],
        'condition':      d['weather'][0]['description'],
        'condition_main': d['weather'][0]['main'],
        'wind_speed_ms':  d['wind']['speed'],
        'pressure_hpa':   d['main']['pressure'],
    }


def fetch_demand_trends(query: str, max_results: int = 3) -> dict:
    # Call the live Tavily API for current market demand signals.
    resp = requests.post(
        'https://api.tavily.com/search',
        json={
            'api_key':        TAVILY_KEY,
            'query':          query,
            'max_results':    max_results,
            'search_depth':   'basic',
            'include_answer': True,
        },
        timeout=15,
    )
    resp.raise_for_status()
    data = resp.json()
    return {
        'answer':  data.get('answer', ''),
        'results': [
            {'title': r['title'], 'content': r['content'][:350]}
            for r in data.get('results', [])
        ],
    }


# Quick live check so we know both external services are working.
print('Verifying live API connections...')
print()

live_weather = fetch_weather(STORE_CITY)
print(f'OpenWeatherMap -- {live_weather["city"]}, {live_weather["country"]}:')
print(f'  Temperature : {live_weather["temperature_c"]} C  (feels like {live_weather["feels_like_c"]} C)')
print(f'  Condition   : {live_weather["condition"]}')
print(f'  Humidity    : {live_weather["humidity_pct"]}%   Wind: {live_weather["wind_speed_ms"]} m/s')

print()
live_demand = fetch_demand_trends(f'retail grocery product demand {STORE_CITY} India')
print(f'Tavily Search -- query returned {len(live_demand["results"])} results:')
if live_demand['answer']:
    print(f'  Synthesised: {live_demand["answer"][:220]}...')
for r in live_demand['results'][:2]:
    print(f'  - {r["title"][:80]}')

Verifying live API connections...

{'coord': {'lon': 77.6033, 'lat': 12.9762}, 'weather': [{'id': 804, 'main': 'Clouds', 'description': 'overcast clouds', 'icon': '04d'}], 'base': 'stations', 'main': {'temp': 27.23, 'feels_like': 28.8, 'temp_min': 27.15, 'temp_max': 29.64, 'pressure': 1009, 'humidity': 65, 'sea_level': 1009, 'grnd_level': 911}, 'visibility': 10000, 'wind': {'speed': 9.12, 'deg': 270, 'gust': 12.2}, 'clouds': {'all': 100}, 'dt': 1785221792, 'sys': {'type': 2, 'id': 2109756, 'country': 'IN', 'sunrise': 1785198846, 'sunset': 1785244671}, 'timezone': 19800, 'id': 1277333, 'name': 'Bengaluru', 'cod': 200}
OpenWeatherMap -- Bengaluru, IN:
  Temperature : 27.2 C  (feels like 28.8 C)
  Condition   : overcast clouds
  Humidity    : 65%   Wind: 9.12 m/s

Tavily Search -- query returned 3 results:
  Synthesised: Bengaluru's grocery market is growing, driven by e-commerce and convenience, with significant expansion by Amazon Fresh and BigBasket. The online grocery market is expand

## Section 1: REST API Integration -- Live External Services

In the REST pattern the developer manually writes a natural-language tool description and the AI uses that description to decide when and how to call the tool.

**Strengths:** Mature ecosystem, no new dependencies, works with any HTTP service.
**Weakness:** Tool schema lives in the developer's prompt. Schema drift and multi-agent reuse require copy-pasting descriptions across every agent that needs the tool.

The two tools below call **real external APIs** on every invocation:
- `get_store_weather` -- live call to OpenWeatherMap
- `search_demand_trends` -- live call to Tavily Search

In [6]:
# This cell shows the REST approach.
# The developer writes the tool schema by hand and the model uses it.
# REST tool schema -- developer writes this by hand for each agent that needs it
REST_TOOLS = [
    {
        'type': 'function',
        'function': {
            'name': 'get_store_weather',
            'description': (
                'Get real-time weather at a Walmart India store location. '
                'Use this to identify weather-driven demand: '
                'rain drives umbrella/raincoat/waterproof footwear sales, '
                'heat drives cold beverages/ice cream/sunscreen sales, '
                'cold drives hot beverages/heaters/blanket sales.'
            ),
            'parameters': {
                'type': 'object',
                'properties': {
                    'city':         {'type': 'string', 'description': 'City where the Walmart store is located'},
                    'country_code': {'type': 'string', 'description': 'ISO country code (default IN for India)'},
                },
                'required': ['city'],
            },
        },
    },
    {
        'type': 'function',
        'function': {
            'name': 'search_demand_trends',
            'description': (
                'Search for real-time retail product demand trends and market signals using Tavily. '
                'Use this to identify high-demand product categories based on current market conditions.'
            ),
            'parameters': {
                'type': 'object',
                'properties': {
                    'query':       {'type': 'string', 'description': 'Search query for demand or market signals'},
                    'max_results': {'type': 'integer', 'description': 'Number of results to return (1-5)'},
                },
                'required': ['query'],
            },
        },
    },
]

# name = "get_store_weather"
# args = {"city": "Bengaluru"}
def execute_rest_tool(name: str, args: dict) -> str:
    # Route each REST tool call to the matching Python function.
    if name == 'get_store_weather':
        result = fetch_weather(args['city'], args.get('country_code', 'IN'))
    elif name == 'search_demand_trends':
        result = fetch_demand_trends(args['query'], args.get('max_results', 3))
    else:
        result = {'error': f'Unknown tool: {name}'}
    return json.dumps(result)


def walmart_rest_agent(query: str) -> dict: # query = "What should we prioritise stocking today?"
    # Run a full REST-style agent loop with live tool calls.
    start    = time.time()
    messages = [
        {
            'role': 'system',
            'content': (
                f'You are an AI assistant for Walmart India store {STORE_ID} in {STORE_CITY}. '
                'Use available tools to retrieve live data before making recommendations. '
                'Base your answer entirely on the real data returned by the tools.'
            ),
        },
        {'role': 'user', 'content': query},
    ]

    tools_called = []
    total_in = total_out = 0

# Ask model
#    ↓
# Call weather tool
#    ↓
# Send weather result back
#    ↓
# Call demand tool
#    ↓
# Send demand result back
#    ↓
# Generate final recommendation

    # Keep calling the model until it stops asking for tools.
    for _ in range(6):
        resp = client.chat.completions.create(
            model='gpt-4o-mini', messages=messages, tools=REST_TOOLS,
            tool_choice='auto', temperature=0, max_tokens=500,
        )
        total_in  += resp.usage.prompt_tokens
        total_out += resp.usage.completion_tokens
        msg = resp.choices[0].message

        if not msg.tool_calls:
            final_answer = msg.content.strip()
            break

        messages.append(msg)
        for tc in msg.tool_calls:
            args   = json.loads(tc.function.arguments)
            result = execute_rest_tool(tc.function.name, args)
            tools_called.append({'tool': tc.function.name, 'args': args})
            messages.append({'role': 'tool', 'tool_call_id': tc.id, 'content': result})

    latency = time.time() - start
    cost    = (total_in * 0.15 + total_out * 0.60) / 1_000_000
    return {
        'answer':       final_answer,
        'tools_called': tools_called,
        'latency_sec':  round(latency, 2),
        'cost_usd':     round(cost, 6),
        'tokens_in':    total_in,
        'tokens_out':   total_out,
        'protocol':     'REST',
    }

In [7]:
# Ask the REST agent the main Walmart store question and inspect the result.
STORE_QUERY = (
    f'I am the store manager at {STORE_ID} in {STORE_CITY}. '
    "Based on today's actual weather conditions and current market demand signals, "
    'give me 3 specific product categories I should prioritise stocking today. '
    'For each category, explain why the live data supports this recommendation.'
)

print(f'Query: {STORE_QUERY}')
print()

rest_result = walmart_rest_agent(STORE_QUERY)

# Show which tools were used before we read the final answer.
print('Tools called by the REST agent:')
for t in rest_result['tools_called']:
    print(f'  {t["tool"]}({json.dumps(t["args"])})')

print()
print('REST Agent Answer (based on live data):')
print('-' * 70)
print(rest_result['answer'])
print('-' * 70)
print(f'Latency : {rest_result["latency_sec"]}s')
print(f'Cost    : ${rest_result["cost_usd"]}')
print(f'Protocol: REST (hand-written tool schema)')

Query: I am the store manager at WMT-2847 in Bengaluru. Based on today's actual weather conditions and current market demand signals, give me 3 specific product categories I should prioritise stocking today. For each category, explain why the live data supports this recommendation.

{'coord': {'lon': 77.6033, 'lat': 12.9762}, 'weather': [{'id': 804, 'main': 'Clouds', 'description': 'overcast clouds', 'icon': '04d'}], 'base': 'stations', 'main': {'temp': 28.14, 'feels_like': 29.9, 'temp_min': 27.84, 'temp_max': 29.64, 'pressure': 1009, 'humidity': 62, 'sea_level': 1009, 'grnd_level': 911}, 'visibility': 10000, 'wind': {'speed': 9.12, 'deg': 270, 'gust': 12.2}, 'clouds': {'all': 100}, 'dt': 1785222772, 'sys': {'type': 2, 'id': 2109756, 'country': 'IN', 'sunrise': 1785198846, 'sunset': 1785244671}, 'timezone': 19800, 'id': 1277333, 'name': 'Bengaluru', 'cod': 200}
Tools called by the REST agent:
  get_store_weather({"city": "Bengaluru", "country_code": "IN"})
  search_demand_trends({"quer

In [8]:
# Drawbacks of the REST approach:

# REST itself is not bad. The problem is how the AI tool integration is managed.

# 1. Tool schema is written manually
# REST_TOOLS = [...]
# When the API changes, this schema must also be updated manually.

# 2. Schema drift can happen
# Suppose the real weather API changes:
# city → location
# but the tool description still expects city.
# Then the LLM may generate an incorrect tool call. The actual API and the AI-facing schema are no longer synchronized.

# 3. Tool definitions get duplicated
# If five different agents need the weather tool, the same description may be copied into all five agents.
# Weather schema
#    ├── Inventory agent
#    ├── Store agent
#    ├── Pricing agent
#    ├── Supply-chain agent
#    └── Customer-service agent
# Any change must then be updated in every place.

# 4. Routing is manually coded
# As the number of tools grows, this routing code becomes larger and harder to maintain.

# 5. Discovery and reuse are limited
# The LLM cannot automatically discover a standard catalogue of available tools. The application must send the manually prepared tool definitions to it.

In [9]:
# Why move to MCP?

# MCP—Model Context Protocol—standardises how tools are exposed to AI applications.

# Instead of every agent maintaining its own copy of the tool schema:
# Multiple agents
#       ↓
# Common MCP server
#       ↓
# Weather and demand tools

# The MCP server becomes the central source of truth.


# Benefits of MCP
# - Tool schemas are defined once and reused.
# - Agents can discover available tools.
# - Changes are maintained centrally.
# - Less schema duplication and drift.
# - Tools can be reused across different models and frameworks.
# - The application needs less manual routing and integration code.

## Section 2: Model Context Protocol (MCP) -- Real FastMCP Server

MCP (Anthropic, 2024) separates tool definition from tool calling.
The MCP server owns the schema. Any MCP-compatible AI client connects, discovers tools automatically, and calls them -- without the developer writing a single word of tool description.

**Architecture:**
```
AI Host (GPT-4o-mini)  <-->  MCP Client (test_client)  <-->  FastMCP Server  <-->  Live External APIs
```

**What changes vs REST:**
- Tool descriptions live on the **server**, not in developer prompts
- Schema is **machine-readable** JSON -- no natural language required
- Any number of AI clients can point at the same server without duplicating schema
- The server handles versioning; clients auto-discover changes on reconnect

**What stays the same:**
- Underlying HTTP calls to OpenWeatherMap and Tavily are identical
- OpenAI token cost per call is the same
- MCP adds a schema-fetch round-trip on first connection (~5ms)

The FastMCP server below uses real API calls inside every registered tool.

In [10]:
# This cell sets up the FastMCP server and registers its tools.
# The main idea is that tool definitions live on the server side.
MCP_AVAILABLE = False

try:
    import inspect
    from mcp.server.fastmcp import FastMCP # used to create the MCP server

    # This helps async-friendly packages behave inside Jupyter.
    try:
        import nest_asyncio
        nest_asyncio.apply()
    except ImportError:
        pass

    # Creating the MCP server
    walmart_mcp = FastMCP(
        'walmart-store-ops',
        instructions=(
            'Walmart India Store Operations MCP Server. '
            'Exposes real-time weather intelligence and market demand signals '
            'for store management decisions at WMT stores across India.'
        ),
    )

    @walmart_mcp.tool()
    def get_store_weather(city: str, country_code: str = 'IN') -> dict:
        """Get current weather at a Walmart India store location for demand planning."""
        return fetch_weather(city, country_code)

    @walmart_mcp.tool()
    def search_demand_trends(query: str, max_results: int = 3) -> dict:
        """Search live retail demand trends and market signals for store planning."""
        return fetch_demand_trends(query, max_results)

    # Registering the store-information tool
    @walmart_mcp.tool()
    def get_store_info(store_id: str) -> dict:
        """Retrieve operational metadata for a Walmart India store."""
        registry = {
            'WMT-2847': {'location': 'Bengaluru, Karnataka', 'format': 'Supercenter',
                         'departments': 32, 'daily_queries': 1200, 'region': 'South India'},
            'WMT-1023': {'location': 'Mumbai, Maharashtra',  'format': 'Supercenter',
                         'departments': 28, 'daily_queries': 1800, 'region': 'West India'},
            'WMT-0511': {'location': 'Delhi NCR',            'format': 'Supercenter',
                         'departments': 35, 'daily_queries': 2100, 'region': 'North India'},
            'WMT-3302': {'location': 'Hyderabad, Telangana', 'format': 'Neighborhood Market',
                         'departments': 18, 'daily_queries':  620, 'region': 'South India'},
        }
        return registry.get(store_id, {'error': f'Store {store_id} not found in registry'})

    def _annotation_to_json_type(annotation) -> str:
        # Convert Python type hints into simple JSON schema types.
        return {
            str: 'string',
            int: 'integer',
            float: 'number',
            bool: 'boolean',
            dict: 'object',
            list: 'array',
        }.get(annotation, 'string')

    def _build_mcp_tool_entry(func) -> dict:
        # Build a small machine-readable schema for each MCP tool.
        sig = inspect.signature(func) # eg: def get_store_weather(city: str, country_code: str = 'IN')
        properties = {}
        required = []

        for param_name, param in sig.parameters.items():
            if param.kind not in (inspect.Parameter.POSITIONAL_OR_KEYWORD, inspect.Parameter.KEYWORD_ONLY):
                continue

            prop = {'type': _annotation_to_json_type(param.annotation)}
            if param.default is not inspect._empty:
                prop['default'] = param.default
            else:
                required.append(param_name)
            properties[param_name] = prop

        return {
            'name': func.__name__,
            'description': inspect.getdoc(func) or '',
            'inputSchema': {
                'type': 'object',
                'properties': properties,
                'required': required,
            },
            'handler': func,
        }

    # Keep the registered MCP tools in one place for easy lookup below.
    MCP_TOOL_REGISTRY = {
        tool['name']: tool
        for tool in [
            _build_mcp_tool_entry(get_store_weather),
            _build_mcp_tool_entry(search_demand_trends),
            _build_mcp_tool_entry(get_store_info),
        ]
    }

    # call_mcp_tool(
    # "get_store_weather",
    # {"city": "Bengaluru"}
    # )
    def call_mcp_tool(name: str, args: dict) -> dict:
        # Run an MCP tool by name using the saved registry entry.
        return MCP_TOOL_REGISTRY[name]['handler'](**args)

    MCP_AVAILABLE = True
    print(f'FastMCP server created: {walmart_mcp.name}')
    print()
    print('Tools registered on the MCP server:')
    print('  - get_store_weather    (live data: OpenWeatherMap API)')
    print('  - search_demand_trends (live data: Tavily Search API)')
    print('  - get_store_info       (store registry -- note: 3 tools vs 2 for REST)')
    print()
    print('Key difference from REST: tool descriptions live here on the server,')
    print('not in the developer\'s agent code. Any MCP client discovers them automatically.')

except ImportError as e:
    print(f'mcp package not installed: {e}')
    print('Install: pip install mcp --break-system-packages')
    print()
    print('The REST section above is fully functional without this package.')
    print('FastMCP cells below will be skipped gracefully.')

FastMCP server created: walmart-store-ops

Tools registered on the MCP server:
  - get_store_weather    (live data: OpenWeatherMap API)
  - search_demand_trends (live data: Tavily Search API)
  - get_store_info       (store registry -- note: 3 tools vs 2 for REST)

Key difference from REST: tool descriptions live here on the server,
not in the developer's agent code. Any MCP client discovers them automatically.


In [11]:
# Show what an MCP client would discover from the server.
# The important point is that the schema comes from the server, not the prompt.
# Step 1: MCP Tool Discovery -- client connects and lists available tools
# This models what an MCP-compatible host receives from the server-owned tool registry.

if MCP_AVAILABLE:
    tools_list = list(MCP_TOOL_REGISTRY.values()) # get_store_weather, search_demand_trends, get_store_info

    print('MCP Protocol: Tool Discovery')
    print(f'Server  : {walmart_mcp.name}')
    print(f'Tools   : {len(tools_list)} (auto-discovered -- no developer prompt required)')
    print()
    for t in tools_list:
        props    = list(t['inputSchema'].get('properties', {}).keys())
        required = t['inputSchema'].get('required', [])
        desc     = t['description'][:110] + '...' if len(t['description']) > 110 else t['description']
        print(f'  name        : {t["name"]}')
        print(f'  description : {desc}')
        print(f'  parameters  : {props}')
        print(f'  required    : {required}')
        print()

    print('Observation: The client discovered the schema from the server.')
    print('No developer wrote tool descriptions in this agent\'s code.')
else:
    print('Skipped: mcp package not installed.')

MCP Protocol: Tool Discovery
Server  : walmart-store-ops
Tools   : 3 (auto-discovered -- no developer prompt required)

  name        : get_store_weather
  description : Get current weather at a Walmart India store location for demand planning.
  parameters  : ['city', 'country_code']
  required    : ['city']

  name        : search_demand_trends
  description : Search live retail demand trends and market signals for store planning.
  parameters  : ['query', 'max_results']
  required    : ['query']

  name        : get_store_info
  description : Retrieve operational metadata for a Walmart India store.
  parameters  : ['store_id']
  required    : ['store_id']

Observation: The client discovered the schema from the server.
No developer wrote tool descriptions in this agent's code.


In [12]:
# Call the MCP tools directly without involving an LLM.
# This helps us separate tool behavior from model behavior.
# Step 2: Direct MCP Tool Execution (no LLM involved)
# The MCP server tools are invoked through the server-owned registry and return real live data.

if MCP_AVAILABLE:
    mcp_w = call_mcp_tool(
        'get_store_weather',
        {'city': STORE_CITY, 'country_code': 'IN'},
    )
    mcp_d = call_mcp_tool(
        'search_demand_trends',
        {'query': f'grocery retail demand trends {STORE_CITY} India', 'max_results': 2},
    )
    mcp_s = call_mcp_tool(
        'get_store_info',
        {'store_id': STORE_ID},
    )

    print('MCP Direct Tool Calls (via MCP server registry -- no LLM involved):')
    print()
    print('get_store_weather:')
    print(f'  {json.dumps(mcp_w)}')
    print()
    print('search_demand_trends:')
    print(f'  answer  : {mcp_d.get("answer", "")[:220]}...')
    print(f'  results : {len(mcp_d.get("results", []))} articles')
    print()
    print('get_store_info:')
    print(f'  {json.dumps(mcp_s)}')
    print()
    print('All three tool calls returned live external data from the MCP server tools.')
else:
    print('Skipped: mcp package not installed.')

{'coord': {'lon': 77.6033, 'lat': 12.9762}, 'weather': [{'id': 804, 'main': 'Clouds', 'description': 'overcast clouds', 'icon': '04d'}], 'base': 'stations', 'main': {'temp': 28.78, 'feels_like': 30.39, 'temp_min': 28.39, 'temp_max': 30.21, 'pressure': 1008, 'humidity': 58, 'sea_level': 1008, 'grnd_level': 911}, 'visibility': 10000, 'wind': {'speed': 9.12, 'deg': 270, 'gust': 12.2}, 'clouds': {'all': 100}, 'dt': 1785223673, 'sys': {'type': 2, 'id': 2109756, 'country': 'IN', 'sunrise': 1785198846, 'sunset': 1785244671}, 'timezone': 19800, 'id': 1277333, 'name': 'Bengaluru', 'cod': 200}
MCP Direct Tool Calls (via MCP server registry -- no LLM involved):

get_store_weather:
  {"city": "Bengaluru", "country": "IN", "temperature_c": 28.8, "feels_like_c": 30.4, "humidity_pct": 58, "condition": "overcast clouds", "condition_main": "Clouds", "wind_speed_ms": 9.12, "pressure_hpa": 1008}

search_demand_trends:
  answer  : Bengaluru's grocery retail demand is growing, driven by increased urbanizat

In [13]:
# Now combine MCP tool schemas, MCP tool execution, and the LLM in one loop.
# This mirrors the REST agent flow, but the tool schema comes from the server side.
# Step 3: Full MCP Agent -- schema from server registry, tool execution from MCP server definitions

if MCP_AVAILABLE:
    def walmart_mcp_agent(query: str) -> dict:
        # Run a full MCP-style agent loop using the server-owned schema.
        start = time.time()

        openai_tools = [
            {
                'type': 'function',
                'function': {
                    'name':        t['name'],
                    'description': t['description'],
                    'parameters':  t['inputSchema'],
                },
            }
            for t in MCP_TOOL_REGISTRY.values()
        ]

        messages = [
            {
                'role': 'system',
                'content': (
                    f'You are an AI assistant for Walmart India store {STORE_ID} in {STORE_CITY}. '
                    'Tools are provided by the Walmart MCP server. '
                    'Use all relevant tools to give a complete, data-driven recommendation.'
                ),
            },
            {'role': 'user', 'content': query},
        ]

        tools_called = []
        total_in = total_out = 0
        final_answer = ''

# LLM asks for store information
#             ↓
# Store result returned
#             ↓
# LLM asks for weather
#             ↓
# Weather result returned
#             ↓
# LLM asks for demand trends
#             ↓
# Demand result returned
#             ↓
# LLM writes final recommendation

        # Keep going until the model stops asking for tool calls.
        for _ in range(6):
            resp = client.chat.completions.create(
                model='gpt-4o-mini', messages=messages, tools=openai_tools,
                tool_choice='auto', temperature=0, max_tokens=500,
            )
            total_in  += resp.usage.prompt_tokens
            total_out += resp.usage.completion_tokens
            msg = resp.choices[0].message

            if not msg.tool_calls:
                final_answer = msg.content.strip()
                break

            messages.append(msg)

            for tc in msg.tool_calls:
                args   = json.loads(tc.function.arguments)
                result = call_mcp_tool(tc.function.name, args)
                tools_called.append({'tool': tc.function.name, 'args': args})
                messages.append({
                    'role':         'tool',
                    'tool_call_id': tc.id,
                    'content':      json.dumps(result),
                })

        latency = time.time() - start
        cost    = (total_in * 0.15 + total_out * 0.60) / 1_000_000
        return {
            'answer':       final_answer,
            'tools_called': tools_called,
            'latency_sec':  round(latency, 2),
            'cost_usd':     round(cost, 6),
            'tokens_in':    total_in,
            'tokens_out':   total_out,
            'protocol':     'MCP',
        }

    mcp_result = walmart_mcp_agent(STORE_QUERY)

    print('Tools called via MCP protocol:')
    for t in mcp_result['tools_called']:
        print(f'  {t["tool"]}({json.dumps(t["args"])})')
    print()
    print('MCP Agent Answer (based on live data):')
    print('-' * 70)
    print(mcp_result['answer'])
    print('-' * 70)
    print(f'Latency : {mcp_result["latency_sec"]}s')
    print(f'Cost    : ${mcp_result["cost_usd"]}')
    print(f'Protocol: MCP (schema auto-discovered from FastMCP server)')

else:
    mcp_result = {**rest_result, 'protocol': 'MCP (fallback -- install mcp package)'}
    print('mcp package not installed. Showing REST result as reference.')
    print(f'Answer: {mcp_result["answer"][:300]}...')

[07/28/26 13:02:35] INFO     HTTP Request: POST https://api.openai.com/v1/chat/completions          ]8;id=274713;file:///opt/homebrew/lib/python3.11/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=902823;file:///opt/homebrew/lib/python3.11/site-packages/httpx/_client.py#1025\1025]8;;\
                             "HTTP/1.1 200 OK"                                                                     

{'coord': {'lon': 77.6033, 'lat': 12.9762}, 'weather': [{'id': 804, 'main': 'Clouds', 'description': 'overcast clouds', 'icon': '04d'}], 'base': 'stations', 'main': {'temp': 28.78, 'feels_like': 30.39, 'temp_min': 28.39, 'temp_max': 30.21, 'pressure': 1008, 'humidity': 58, 'sea_level': 1008, 'grnd_level': 911}, 'visibility': 10000, 'wind': {'speed': 9.12, 'deg': 270, 'gust': 12.2}, 'clouds': {'all': 100}, 'dt': 1785223673, 'sys': {'type': 2, 'id': 2109756, 'country': 'IN', 'sunrise': 1785198846, 'sunset': 1785244671}, 'timezone': 19800, 'id': 1277333, 'name': 'Bengaluru', 'cod': 200}


[07/28/26 13:02:39] INFO     HTTP Request: POST https://api.openai.com/v1/chat/completions          ]8;id=693369;file:///opt/homebrew/lib/python3.11/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=515770;file:///opt/homebrew/lib/python3.11/site-packages/httpx/_client.py#1025\1025]8;;\
                             "HTTP/1.1 200 OK"                                                                     

[07/28/26 13:02:49] INFO     HTTP Request: POST https://api.openai.com/v1/chat/completions          ]8;id=967861;file:///opt/homebrew/lib/python3.11/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=661543;file:///opt/homebrew/lib/python3.11/site-packages/httpx/_client.py#1025\1025]8;;\
                             "HTTP/1.1 200 OK"                                                                     

Tools called via MCP protocol:
  get_store_weather({"city": "Bengaluru"})
  search_demand_trends({"query": "Bengaluru", "max_results": 3})
  search_demand_trends({"query": "weather related products", "max_results": 3})

MCP Agent Answer (based on live data):
----------------------------------------------------------------------
Based on the current weather conditions in Bengaluru and the latest market demand signals, here are three specific product categories you should prioritize stocking today:

1. **Beverages (Cold Drinks and Juices)**
   - **Reasoning**: The current temperature in Bengaluru is around 28.8°C with a humidity level of 58%. These conditions typically lead to increased demand for cold beverages as people look to stay refreshed. The overcast weather may also encourage people to enjoy indoor activities, where cold drinks are often consumed. Stocking a variety of cold drinks, including juices and soft drinks, will cater to this demand.

2. **Snacks (Chips and Ready-to-Eat 

In [14]:
# Compare the two protocols side by side using the live results from this run.
print(f'{"Dimension":<44} {"REST API":>22} {"MCP (FastMCP)":>22}')
print('=' * 90)

rows = [
    ('Latency (sec)',
     f'{rest_result["latency_sec"]:.2f}',
     f'{mcp_result["latency_sec"]:.2f}' if MCP_AVAILABLE else 'N/A'),
    ('Cost per query ($)',
     f'{rest_result["cost_usd"]:.6f}',
     f'{mcp_result["cost_usd"]:.6f}' if MCP_AVAILABLE else 'N/A'),
    ('Tools called',
     str(len(rest_result['tools_called'])),
     str(len(mcp_result['tools_called'])) if MCP_AVAILABLE else 'N/A'),
    ('Tool schema source',    'Hand-written in agent code',       'Auto-discovered from MCP server'),
    ('Schema owner',          'Developer (copy-paste per agent)', 'MCP server (single source of truth)'),
    ('Schema evolution',      'Edit every agent that uses it',    'Server bumps version, clients refresh'),
    ('Multi-agent reuse',     'Rewrite schema per client',        'Any client points at same server'),
    ('Auth handling',         'Per-integration (custom code)',    'Built into MCP transport layer'),
    ('Protocol maturity',     'Decades (HTTP/REST standard)',     '2024 (Anthropic open standard)'),
    ('Best for',              'Existing API, single client',      'Multi-agent platform, tool registry'),
]

for label, rest_val, mcp_val in rows:
    print(f'{label:<44} {rest_val:>22} {mcp_val:>22}')

print()
print('Conclusion: For 50,000+ daily queries across 4,700 stores with multiple')
print('agent types, MCP eliminates tool description duplication that makes')
print('REST-only architectures unscalable at enterprise.')

Dimension                                                  REST API          MCP (FastMCP)
Latency (sec)                                                 12.63                  18.73
Cost per query ($)                                         0.000443               0.000563
Tools called                                                      2                      3
Tool schema source                           Hand-written in agent code Auto-discovered from MCP server
Schema owner                                 Developer (copy-paste per agent) MCP server (single source of truth)
Schema evolution                             Edit every agent that uses it Server bumps version, clients refresh
Multi-agent reuse                            Rewrite schema per client Any client points at same server
Auth handling                                Per-integration (custom code) Built into MCP transport layer
Protocol maturity                            Decades (HTTP/REST standard) 2024 (Anthropic open 

In [15]:
# To be continued...